# Восстановление рекомендательного проекта из S3

**Назначение:** восстановление parquet файлов из s3.

Уже обученные модели ALS и CatBoost **не нужны для выдачи** ранее рассчитанных рекомендаций. Если `models/catboost_ranker.cbm` не загружали в S3, восстановить веса из Parquet-рекомендаций нельзя; переобучение — отдельная задача.

Запускать из корня `mle-project-sprint-4-v001`, где лежит `.env`. Не коммитить `.env`, ключи или большие данные в Git.

## 1. Зависимости и пути

In [1]:
# Подключаем библиотеки и определяем пути относительно корня проекта
# Если недостаёт boto3: %pip install boto3

import os
from pathlib import Path

import boto3
import pandas as pd
import pyarrow.parquet as pq
import pyarrow.dataset as ds
from botocore.exceptions import ClientError

PROJECT_DIR = Path.cwd()
ENV_PATH = PROJECT_DIR / ".env"
S3_ENDPOINT = "https://storage.yandexcloud.net"

print("Проект:", PROJECT_DIR)
print(".env найден:", ENV_PATH.is_file())

Проект: C:\Users\a.dedyulya\PycharmProjects\mle-project-sprint-4-v001
.env найден: True


## 2. Конфигурация S3 из `.env`

In [2]:
# Загружаем переменные из .env без дополнительных библиотек
# Не печатаем секреты и не перезаписываем уже заданные переменные окружения

if ENV_PATH.exists():
    for line in ENV_PATH.read_text(encoding="utf-8-sig").splitlines():
        line = line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, value = line.split("=", 1)
        key, value = key.strip(), value.strip().strip('"').strip("'")
        os.environ.setdefault(key, value)

required = ["S3_BUCKET_NAME", "AWS_ACCESS_KEY_ID", "AWS_SECRET_ACCESS_KEY"]
missing = [key for key in required if not os.getenv(key)]
if missing:
    raise RuntimeError(f"Нет значений в окружении/.env: {', '.join(missing)}")

bucket_name = os.environ["S3_BUCKET_NAME"]
s3 = boto3.client(
    "s3",
    endpoint_url=S3_ENDPOINT,
    aws_access_key_id=os.environ["AWS_ACCESS_KEY_ID"],
    aws_secret_access_key=os.environ["AWS_SECRET_ACCESS_KEY"],
)

# Проверяем доступ к бакету (секреты не отображаем)
s3.head_bucket(Bucket=bucket_name)
print("Доступ к бакету подтверждён:", bucket_name)

Доступ к бакету подтверждён: s3-student-mle-20260525-1c9dbad81c-freetrack


## 3. Состав обязательных артефактов

In [3]:
# Сопоставляем ключи S3 с локальными путями проекта

ARTIFACTS = {
    "recsys/data/items.parquet": "items.parquet",
    "recsys/data/events.parquet": "events.parquet",
    "recsys/recommendations/top_popular.parquet": "recommendations/top_popular/top_popular.parquet",
    "recsys/recommendations/personal_als.parquet": "recommendations/personal_als/personal_als.parquet",
    "recsys/recommendations/similar.parquet": "recommendations/similar/similar.parquet",
    "recsys/recommendations/recommendations.parquet": "recommendations/recommendations.parquet",
}

remote_manifest = []
for key, local in ARTIFACTS.items():
    try:
        metadata = s3.head_object(Bucket=bucket_name, Key=key)
        remote_manifest.append({
            "s3_key": key,
            "local_path": local,
            "bytes": metadata["ContentLength"],
            "status": "ok",
        })
    except ClientError as exc:
        error_code = exc.response.get("Error", {}).get("Code", "unknown")
        remote_manifest.append({
            "s3_key": key,
            "local_path": local,
            "bytes": None,
            "status": f"error: {error_code}",
        })

display(pd.DataFrame(remote_manifest))
if any(record["status"] != "ok" for record in remote_manifest):
    print("Внимание: некоторые файлы недоступны. Проверьте ключи S3 перед скачиванием.")

,s3_key,local_path,bytes,status
0,recsys/data/items.parquet,items.parquet,18982610,ok
1,recsys/data/events.parquet,events.parquet,1049697887,ok
2,recsys/recommendations/top_popular.parquet,recommendations/top_popular/top_popular.parquet,4042,ok
3,recsys/recommendations/personal_als.parquet,recommendations/personal_als/personal_als.parquet,879307655,ok
4,recsys/recommendations/similar.parquet,recommendations/similar/similar.parquet,858116277,ok
5,recsys/recommendations/recommendations.parquet,recommendations/recommendations.parquet,398737766,ok


## 4. Скачать файлы (повторный запуск безопасен)

In [4]:
# Скачиваем только отсутствующие или отличающиеся по размеру файлы
# Временный .part переименовываем только после успешного скачивания

for record in remote_manifest:
    if record["status"] != "ok":
        print("Пропуск отсутствующего объекта:", record["s3_key"])
        continue

    destination = PROJECT_DIR / record["local_path"]
    destination.parent.mkdir(parents=True, exist_ok=True)

    if destination.exists() and destination.stat().st_size == record["bytes"]:
        print("Уже на месте:", destination.relative_to(PROJECT_DIR))
        continue

    temporary_path = destination.with_name(destination.name + ".part")
    print("Загружаем:", record["s3_key"])
    s3.download_file(bucket_name, record["s3_key"], str(temporary_path))
    if temporary_path.stat().st_size != record["bytes"]:
        raise IOError(f"Размер загруженного файла не совпадает: {destination}")
    temporary_path.replace(destination)
    print("Готово:", destination.relative_to(PROJECT_DIR))

print("Синхронизация завершена")

Загружаем: recsys/data/items.parquet
Готово: items.parquet
Загружаем: recsys/data/events.parquet
Готово: events.parquet
Загружаем: recsys/recommendations/top_popular.parquet
Готово: recommendations\top_popular\top_popular.parquet
Загружаем: recsys/recommendations/personal_als.parquet
Готово: recommendations\personal_als\personal_als.parquet
Загружаем: recsys/recommendations/similar.parquet
Готово: recommendations\similar\similar.parquet
Загружаем: recsys/recommendations/recommendations.parquet
Готово: recommendations\recommendations.parquet
Синхронизация завершена


## 5. Проверка размеров и схем без загрузки огромных таблиц в RAM

In [5]:
# Проверяем число строк, колонки и размер каждого parquet по метаданным

summary = []
for record in remote_manifest:
    path = PROJECT_DIR / record["local_path"]
    if not path.is_file():
        summary.append({"file": record["local_path"], "status": "missing"})
        continue
    pf = pq.ParquetFile(path)
    summary.append({
        "file": record["local_path"],
        "status": "ok",
        "rows": pf.metadata.num_rows,
        "row_groups": pf.metadata.num_row_groups,
        "columns": ", ".join(pf.schema_arrow.names),
        "size_gb": round(path.stat().st_size / (1024 ** 3), 3),
    })

display(pd.DataFrame(summary))

,file,status,rows,row_groups,columns,size_gb
0,items.parquet,ok,1000000,1,"track_id, albums, artists, genres",0.018
1,events.parquet,ok,222629898,213,"user_id, track_id, track_seq, started_at",0.978
2,recommendations/top_popular/top_popular.parquet,ok,100,1,"track_id, score, rank",0.000
3,recommendations/personal_als/personal_als.parquet,ok,134256600,135,"user_id, track_id, score, rank",0.819
4,recommendations/similar/similar.parquet,ok,99969500,100,"track_id, similar_track_id, score, rank",0.799
5,recommendations/recommendations.parquet,ok,134256600,135,"user_id, track_id, score, rank",0.371


In [6]:
# Проверяем обязательные колонки и смотрим первые 5 строк без полного read_parquet

EXPECTED_COLUMNS = {
    "items.parquet": {"track_id", "albums", "artists", "genres"},
    "events.parquet": {"user_id", "track_id", "track_seq", "started_at"},
    "recommendations/top_popular/top_popular.parquet": {"track_id", "score", "rank"},
    "recommendations/personal_als/personal_als.parquet": {"user_id", "track_id", "score", "rank"},
    "recommendations/similar/similar.parquet": {"track_id", "similar_track_id", "score", "rank"},
    "recommendations/recommendations.parquet": {"user_id", "track_id", "score", "rank"},
}

for relative_path, expected in EXPECTED_COLUMNS.items():
    path = PROJECT_DIR / relative_path
    if not path.exists():
        print("ОТСУТСТВУЕТ:", relative_path)
        continue
    pf = pq.ParquetFile(path)
    missing_columns = expected.difference(pf.schema_arrow.names)
    print(f"{relative_path}: {'OK' if not missing_columns else 'нет колонок ' + str(missing_columns)}")
    preview = next(pf.iter_batches(batch_size=5)).to_pandas()
    display(preview.head())

items.parquet: OK


,track_id,albums,artists,genres
0,26,"[3, 2490753]",[16],"[11, 21]"
1,38,"[3, 2490753]",[16],"[11, 21]"
2,135,"[12, 214, 2490809]",[84],[11]
3,136,"[12, 214, 2490809]",[84],[11]
4,138,"[12, 214, 322, 72275, 72292, 91199, 213505, 24...",[84],[11]


events.parquet: OK


,user_id,track_id,track_seq,started_at
0,0,99262,1,2022-07-17
1,0,589498,2,2022-07-19
2,0,590262,3,2022-07-21
3,0,590303,4,2022-07-22
4,0,590692,5,2022-07-22


recommendations/top_popular/top_popular.parquet: OK


...

,user_id,track_id,score,rank
0,0,597196,0.070913,1
1,0,19972786,0.070438,2
2,0,6006252,0.069134,3
3,0,17856052,0.068309,4
4,0,25903468,0.066484,5


recommendations/similar/similar.parquet: OK


,track_id,similar_track_id,score,rank
0,26,20111365,0.894730,1
1,26,21644998,0.873237,2
2,26,4850799,0.865434,3
3,26,21846409,0.860117,4
4,26,22836932,0.857655,5


recommendations/recommendations.parquet: OK


,user_id,track_id,score,rank
0,0,10695539,0.366669,1
1,0,16202171,0.366669,2
2,0,19972786,0.268634,3
3,0,17856052,0.268634,4
4,0,22434771,0.268634,5


## 6. Проверяем выборочную выдачу рекомендаций для сервиса

In [7]:
# Тестируем пользовательскую выборку: чтение всех 134 млн рекомендаций в pandas не требуется

recommendations_dataset = ds.dataset(
    PROJECT_DIR / "recommendations/recommendations.parquet",
    format="parquet",
)

user_id = 0
user_recommendations = (
    recommendations_dataset.to_table(
        columns=["user_id", "track_id", "score", "rank"],
        filter=ds.field("user_id") == user_id,
    )
    .to_pandas()
    .sort_values("rank")
    .head(10)
)

display(user_recommendations)
print("Рекомендаций для пользователя:", len(user_recommendations))
# Примечание: без партиционирования dataset может сканировать значительную часть файла.

,user_id,track_id,score,rank
0,0,10695539,0.366669,1
1,0,16202171,0.366669,2
2,0,19972786,0.268634,3
3,0,17856052,0.268634,4
4,0,22434771,0.268634,5
5,0,15397485,0.268634,6
6,0,17973078,0.268634,7
7,0,23419375,0.268634,8
8,0,21448396,0.268634,9
9,0,14697223,0.268634,10


Рекомендаций для пользователя: 10


## 7. Проверяем, сохранились ли файлы обученных моделей

Ранее `CatBoostRanker` сохранялся локально в `models/catboost_ranker.cbm`, **но мы не загружали этот файл в S3 в обязательных шагах**. Сами готовые офлайн-рекомендации не требуют модели при запуске API.

Если модель оказалась в S3 по ключу `recsys/models/catboost_ranker.cbm`, можно восстановить её. Это **не гарантированный** ключ, а вариант для проверки. Другая модель (ALS) также не входила в обязательные артефакты S3.

In [8]:
# Пытаемся восстановить CatBoostRanker, только если такой объект действительно есть

model_key = "recsys/models/catboost_ranker.cbm"
model_path = PROJECT_DIR / "models/catboost_ranker.cbm"

try:
    model_metadata = s3.head_object(Bucket=bucket_name, Key=model_key)
except ClientError as exc:
    code = exc.response.get("Error", {}).get("Code", "unknown")
    print(f"Модель не найдена по ключу {model_key} ({code}). Для сервиса это не блокер.")
else:
    model_path.parent.mkdir(parents=True, exist_ok=True)
    if not model_path.exists() or model_path.stat().st_size != model_metadata["ContentLength"]:
        s3.download_file(bucket_name, model_key, str(model_path))
    print("Модель восстановлена:", model_path)

Модель не найдена по ключу recsys/models/catboost_ranker.cbm (404). Для сервиса это не блокер.


## 8. Что делать дальше

1. Если все шесть Parquet-файлов доступны и проходят проверку, можно продолжить разработку `recommendations_service.py` **без переобучения** ALS/CatBoost.
2. Если нужен именно `models/catboost_ranker.cbm`, потребуется восстановить его из другого бэкапа или отдельно переобучить CatBoost на кандидатах с метками за **16–23 декабря 2022 года**. Период **24–31 декабря** оставить только для финальной оценки.
3. Если нужен сам ALS, его веса нельзя восстановить только по `personal_als.parquet` и `similar.parquet`: потребуется повторное обучение на `events` до **16 декабря 2022 года**.
4. Если отсутствует какой-то Parquet-артефакт, не переобучайте всё подряд: сначала определите, какой именно этап его порождает.

**Важно:** ноутбук, код предыдущих вычислений и сохранённые выводы Jupyter не восстанавливаются из Parquet. Сохраняйте локальный `recommendations.ipynb` и проверяйте `.gitignore` перед GitHub commit.